# SIM V3 · Outdoor · Phase D — validate (g2) + export ONNX

**g2 gate:** tiled surrogate prediction vs a *fresh* FDTD solve on a **held-out** transmitter — envelope dB RMSE, Spearman, complex coherence. Then export the ONNX + browser contract.

In [ ]:
# --- locate SIM V3 (works locally and on Colab) ---
# Colab: clone the repo, then set REPO_ROOT to it, e.g. '/content/Indoor_Walk_Test_7-7'.
REPO_ROOT = ''
import os, sys
if REPO_ROOT:
    SIMV3 = os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')
else:
    SIMV3, d = os.path.abspath('..'), os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, '_bootstrap.py')): SIMV3 = d; break
        c = os.path.join(d, 'Physics Engine', '2D', 'SIM V3')
        if os.path.exists(os.path.join(c, '_bootstrap.py')): SIMV3 = c; break
        d = os.path.dirname(d)
assert os.path.exists(os.path.join(SIMV3, '_bootstrap.py')), f'set REPO_ROOT; not found: {SIMV3}'
sys.path.insert(0, SIMV3); os.chdir(SIMV3)
print('SIM V3 =', SIMV3)

In [ ]:
# Colab only: install deps (skip locally). torch usually preinstalled on Colab GPU.
# !pip -q install numpy scipy matplotlib tqdm onnxruntime
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

### Validate + export

In [ ]:
import fw_unet2d, fw_dataset, fw_infer, numpy as np, _bootstrap as B
from scipy.stats import spearmanr
from bands_v3 import get
model = fw_unet2d.load_model('fw_unet2d_outdoor.pt'); band = get('NR_n41_2506')
city = np.load(B.CITY_DIR / 'material_grid.npy'); cs = 1.0
air = np.argwhere(city[:, 2, :] == 0)
tc = air[np.random.default_rng(7).integers(len(air))]   # held-out tower
p, Ut = fw_dataset._outdoor_field(city, cs, band, (tc[0]*cs, tc[1]*cs),
                                  2.0, 8, 40, 2.5, 1_200_000)
Up = fw_infer.tiled_predict(model, p.classes, p.tx_idx, p.h_m, band.f_mhz)
m = (p.classes == 0)
et = fw_infer._db(np.abs(Ut), np.percentile(np.abs(Ut)[m], 99))
ep = fw_infer._db(np.abs(Up), np.percentile(np.abs(Up)[m], 99)); mm = m & (et > -60)
print('envelope RMSE %.2f dB  Spearman %+.3f'
      % (np.sqrt(np.mean((et[mm]-ep[mm])**2)), spearmanr(et[mm], ep[mm])[0]))
import fw_export
fw_export.export_onnx(model, fw_export.WEB / 'fw_unet2d_outdoor.onnx')
print('exported outdoor ONNX')

### View the comparison

In [ ]:
# (outdoor plots its metrics above; add a side-by-side imshow of et/ep if desired)